# Notebook 05: CUPED

### Purpose
Determines how much precision can be gained by adjusting for pre-experiment covariates, and whether the available covariates in this dataset are useful for variance reduction.

### Objectives
- Whether `history` and `recency` meaningfully reduce variance in visit, conversion, and spend outcomes.
- How single-covariate CUPED compares to a multi-covariate regression adjustment.
- What the results imply about the conditions under which CUPED is and isn't worth applying.

**Note:** Selecting recency and history for CUPED because logically they could have some impact on the outcome metrics.  They appear to be pre-experiment covariates based on the dataset description, but in a production setting measurement date should be explicitly verified relative to experiment start.

In [66]:
import pandas as pd
import numpy as np

from scipy.stats import ttest_ind
import statsmodels.formula.api as smf


In [36]:
hillstrom_df = pd.read_csv("../data/raw/hillstrom.csv")

display(hillstrom_df.info())

display(hillstrom_df.head(), hillstrom_df.tail())

<class 'pandas.DataFrame'>
RangeIndex: 64000 entries, 0 to 63999
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   recency          64000 non-null  int64  
 1   history_segment  64000 non-null  str    
 2   history          64000 non-null  float64
 3   mens             64000 non-null  int64  
 4   womens           64000 non-null  int64  
 5   zip_code         64000 non-null  str    
 6   newbie           64000 non-null  int64  
 7   channel          64000 non-null  str    
 8   segment          64000 non-null  str    
 9   visit            64000 non-null  int64  
 10  conversion       64000 non-null  int64  
 11  spend            64000 non-null  float64
dtypes: float64(2), int64(6), str(4)
memory usage: 5.9 MB


None

,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
63995,10,2) $100 - $200,105.54,1,0,Urban,0,Web,Mens E-Mail,0,0,0.0
63996,5,1) $0 - $100,38.91,0,1,Urban,1,Phone,Mens E-Mail,0,0,0.0
63997,6,1) $0 - $100,29.99,1,0,Urban,1,Phone,Mens E-Mail,0,0,0.0
63998,1,5) $500 - $750,552.94,1,0,Surburban,1,Multichannel,Womens E-Mail,0,0,0.0
63999,1,4) $350 - $500,472.82,0,1,Surburban,0,Web,Mens E-Mail,0,0,0.0


In [80]:
print("Pearson's correlation:")

hillstrom_df[["history", "recency", "visit", "conversion", "spend"]].corr().loc[["history", "recency"], ["visit", "conversion", "spend"]]

Pearson's correlation:


,visit,conversion,spend
history,0.065153,0.029405,0.021729
recency,-0.074765,-0.024412,-0.016348


**Low correlation indicates CUPED will have a very modest variance reduction**

### CUPED on visit, conversion, and spend using history and recency as single-factor covariates

In [ ]:
# CUPED on visit, conversion, and spend using history and recency as single-factor covariates

results = []

for covariate in ["history", "recency"]:

    for metric in ["visit", "conversion", "spend"]:
    
        theta = np.cov(hillstrom_df[metric], hillstrom_df[covariate])[0,1] / np.var(hillstrom_df[covariate], ddof=1)

        hillstrom_df[metric + '_cuped_' + covariate] = hillstrom_df[metric] - theta * (hillstrom_df[covariate] - hillstrom_df[covariate].mean())

        percent_var_reduction = (1 - hillstrom_df[metric + '_cuped_' + covariate].var() / hillstrom_df[metric].var()) * 100

        n_savings = hillstrom_df.shape[0] * (percent_var_reduction / 100)

        results.append((covariate, metric, percent_var_reduction, n_savings))

pd.DataFrame(results, columns=["covariate", "metric", "percent_var_reduction", "n_savings"]) 

,covariate,metric,percent_var_reduction,n_savings
0,history,visit,0.424496,271.677455
1,history,conversion,0.086464,55.336797
2,history,spend,0.047215,30.217565
3,recency,visit,0.558986,357.750786
4,recency,conversion,0.059596,38.141290
5,recency,spend,0.026726,17.104941


- **Applying CUPED means the visits and conversions data is no longer a binary outcome and statistical inference would need to be done with a t-test.**
- **That's no longer an apples to apples comparison with the z test for proportions done for the non-CUPED data.** 
- **Comparing the reduction in variance and thereby reduction in users needed to detect the same effect is the cleanest approach for these two metrics.**

### Compare the original and CUPED-adjusted spend between the two segments using t-tests

In [72]:
outcomes = ["spend", "spend_cuped_history", "spend_cuped_recency"]
results = []

for outcome in outcomes:
    group1 = hillstrom_df.loc[hillstrom_df["segment"] == "Mens E-Mail", outcome]
    group2 = hillstrom_df.loc[hillstrom_df["segment"] == "No E-Mail", outcome]
    tstat, pvalue = ttest_ind(group1, group2, equal_var=False)
    results.append({"outcome": outcome, "t-stat": tstat, "p-value": pvalue})

pd.DataFrame(results)


,outcome,t-stat,p-value
0,spend,5.300140,1.163815e-07
1,spend_cuped_history,5.284132,1.270227e-07
2,spend_cuped_recency,5.312425,1.088061e-07


### CUPED with multiple covariates

In [ ]:
results = []

for metric in ["visit", "conversion", "spend"]:
    
    # Fit a regression of outcome on both covariates
    model = smf.ols(f"{metric} ~ history + recency", data=hillstrom_df).fit()
    # Subtract the predicted values
    hillstrom_df[f"{metric}_cuped_multiple"] = hillstrom_df[metric] - model.fittedvalues

    # This should be the same as the R^2, just a sanity check
    percent_var_reduction = (1 - hillstrom_df[f"{metric}_cuped_multiple"].var() / hillstrom_df[metric].var()) * 100

    results.append({
        "metric": metric,
        "r_squared_pct": model.rsquared * 100,
        "percent_var_reduction": percent_var_reduction,
    })

pd.DataFrame(results)

,metric,r_squared_pct,percent_var_reduction
0,visit,0.791362,0.791362
1,conversion,0.117822,0.117822
2,spend,0.060075,0.060075


### Compare the original and CUPED multiple covariate adjusted spend between the two segments using t-tests

In [79]:
outcomes = ["spend", "spend_cuped_multiple"]
results = []

for outcome in outcomes:
    group1 = hillstrom_df.loc[hillstrom_df["segment"] == "Mens E-Mail", outcome]
    group2 = hillstrom_df.loc[hillstrom_df["segment"] == "No E-Mail", outcome]
    tstat, pvalue = ttest_ind(group1, group2, equal_var=False)
    results.append({"outcome": outcome, "t-stat": tstat, "p-value": pvalue})

pd.DataFrame(results)


,outcome,t-stat,p-value
0,spend,5.300140,1.163815e-07
1,spend_cuped_multiple,5.295054,1.196651e-07


## Summary

- CUPED produced negligible variance reduction across all three outcomes — under 1% for visit, under 0.2% for conversion, under 0.1% for spend.
- Adding a second covariate via regression produced no additional improvement.
- The weak results were expected: `history` and `recency` are customer characteristics, not prior measurements of the outcome. Correlations were under 0.08 in all cases.
- CUPED literature indicates it works best when the covariate is a prior measurement of the same metric - e.g. last month's spend predicting this month's spend.
- **The practical implication: CUPED's value depends on continuous pre-experiment logging. Teams that log outcome metrics before experiments start get outsized returns from variance reduction techniques.**